In [37]:
import pandas
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from peft import LoraConfig, get_peft_model

dataset_path = r"../../datasets/sentiment_dataset.csv"

In [38]:
dataset = pandas.read_csv(dataset_path)

In [39]:
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset["label"])

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [40]:
checkpoint = "vinai/phobert-base" 

In [41]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [54]:
print(model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): RobertaForSequenceClassification(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(64001, 768, padding_idx=1)
          (position_embeddings): Embedding(258, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): Mo

In [42]:
# PhoBERT (RoBERTa-style) requires fixed max_length to avoid shape mismatches.
max_length = 256

def tokenizer_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
        return_token_type_ids=False,
    )


def preprocessing(ds):
    ds = ds.map(tokenizer_fn, batched=True, remove_columns=["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

In [43]:
print(train_dataset['text'][0])

 Đến lần muốn đến thêm lần ngon tuyệt


In [44]:
tokenized_train = preprocessing(train_dataset)
tokenized_test = preprocessing(test_dataset)

Map: 100%|██████████| 882/882 [00:00<00:00, 2027.60 examples/s]


In [45]:
# target_modules = ["q_proj", "v_proj"]

## WARNING: Bert không dùng q_proj,v_proj, 
# Sử dụng ["query", "value"] 
# https://medium.com/@simon.gsponer/a-comprehensive-guide-ii-finetuning-a-bert-llm-with-lora-and-make-it-pipeline-compatible-9508e3822907

target_modules = ["query", "value"]

In [46]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,  
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,181,954 || all params: 136,181,764 || trainable%: 0.8679


In [47]:
from transformers import Trainer, TrainingArguments

In [48]:
training_args = TrainingArguments(
    output_dir="../../models/lora_phobert_lora_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",   # Tránh bị hỏi login wandb trong Colab
    disable_tqdm=False,
)


In [49]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer)

In [50]:
import numpy as np

def compute_metrics(eval_pred):
    """Return metrics dict for Hugging Face Trainer.

    Computes:
    - accuracy
    - f1 (binary if 2 labels, weighted otherwise)
    """
    logits, labels = eval_pred
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    labels = np.array(labels)

    acc = accuracy_score(labels, preds)

    num_labels = logits.shape[-1] if hasattr(logits, "shape") and len(logits.shape) > 1 else None
    if num_labels == 2:
        f1 = f1_score(labels, preds, average="binary", pos_label=1, zero_division=0)
    else:
        f1 = f1_score(labels, preds, average="weighted", zero_division=0)

    return {"accuracy": acc, "f1": f1}

In [51]:
# Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
 )

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [52]:
train_result = trainer.train()
print(f"Final training loss: {train_result.training_loss:.4f}")

eval_metrics = trainer.evaluate(eval_dataset=tokenized_test)
print("Eval metrics:")
for k, v in eval_metrics.items():
    try:
        print(f"  {k}: {float(v):.6f}")
    except Exception:
        print(f"  {k}: {v}")


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.212700,0.224614,0.909297,0.927007
2,0.261200,0.184429,0.927438,0.940741
3,0.188700,0.182982,0.929705,0.942804


Final training loss: 0.2397


Eval metrics:
  eval_loss: 0.182982
  eval_accuracy: 0.929705
  eval_f1: 0.942804
  eval_runtime: 6.760900
  eval_samples_per_second: 130.457000
  eval_steps_per_second: 8.283000
  epoch: 3.000000


In [53]:
import json
import os

model_output_dir = "../../output/phobert_lora_sentiment"
os.makedirs(model_output_dir, exist_ok=True)

# Lưu eval metrics (có eval_loss, eval_accuracy, eval_f1, ...)
metrics_path = os.path.join(model_output_dir, "eval_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({k: float(v) if hasattr(v, "__float__") else v for k, v in eval_metrics.items()}, f, ensure_ascii=False, indent=2)

model.save_pretrained(model_output_dir)
tokenizer.save_pretrained(model_output_dir)
print(f"Saved model to: {model_output_dir}")
print(f"Saved metrics to: {metrics_path}")

Saved model to: ../../output/phobert_lora_sentiment
Saved metrics to: ../../output/phobert_lora_sentiment\eval_metrics.json
